# 04 Molecular Ratios

> Public source code; private observational inputs are not included. Outputs shown here were generated during the original thesis; the adapted notebooks have not been rerun against the raw data.


In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the project root from the working directory or an ancestor."""
    location = Path(start or Path.cwd()).resolve()
    for candidate in (location, *location.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory. The README.md and notebooks/ folder must exist.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
SPECTRA_DIR = PROJECT_ROOT / "spectra"
FIGURES_DIR = PROJECT_ROOT / "figures"
SAVE_FIGURES = False  # Figures render inline; optional exports disabled by default.
SAVE_TABLES = True    # Intermediate tables are required by downstream notebooks.
# Output folders are part of the repository layout; raw data remains private.


> **Archived tables:** The results below are preserved from the original notebook executions and displayed without numerical alteration. **Scientific consistency note:** the archived HCN/HCO⁺ column and the numerator/denominator used in its code require review; do not assume all table labels match the implemented calculation. See `docs/REPRODUCIBILITY.md`.


# Necesito calcular la masa molecular del CO, el ratio trazadores densos/CO y el ratio de gas termalizado R21/10

Cargo los paquetes y los archivos con los resultados del flujo en unidades de $I = \int T_\text{mb}\ dv$ ($\mathrm{K \ km \ s^{-1}}$) 

In [ ]:
import numpy as np
import os
import pandas as pd

results_dir = str(RESULTS_DIR)

# Cargar los datos de los archivos
co_ALMA = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCO/CO_ALMA_beam30m_Jy_K.dat'), sep="\t") #CO(1-0) con su beam y CO (2-1) con el suyo 
co10_ALMA_beam = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCO/CO10_ALMA_beam30m_Jy_K.dat'), sep="\t") # CO(1-0) con beam de CO(2-1) y de densa
co10_IRAM = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_CO10_moment_results.dat'), sep="\t") # CO(1-0) de class
co21_IRAM = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_CO21_moment_results.dat'), sep="\t") # CO(2-1) de class
co21_ALMA_beam10 = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCO/CO21_ALMA_beam30m_Jy_K.dat'), sep="\t") # CO(2-1) con beam de CO(1-0)
hcn = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HCN_moment_results.dat'), sep="\t") # HCN de class
hco = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HCO_moment_results.dat'), sep="\t") # HCO de class
hnc_ul = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosHNC/upper_limit_HNC_2_results.dat'), sep="\t") # HNC de class (solo límite superior) I < 3 * rms * \sqrt{\DeltaV * \deltaV} 
hnc = pd.read_csv(str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HNC_moment_results.dat'), sep="\t") # HNC CEN de class (solo se detecta en el centro)


co10_ALMA = co_ALMA[["fuente", "I_CO10_ALMA_K", "err_I_CO10_ALMA_K"]] # CO(1-0) con su beam
co21_ALMA = co_ALMA[["fuente", "I_CO21_ALMA_K", "err_I_CO21_ALMA_K"]] # CO(2-1) con su beam
co10_beam21 = co10_ALMA_beam[["fuente", "I_CO10_21_ALMA_K", "err_I_CO10_21_ALMA_K"]] # CO(1-0) con beam de CO(2-1)
co10_beamdense = co10_ALMA_beam[["fuente", "I_CO10_DENSE_ALMA_K", "err_I_CO10_DENSE_ALMA_K"]] # CO(1-0) con beam de densa
co10_IRAM = co10_IRAM[["fuente", "I_manual", "SigmaI"]] # CO(1-0) de class
co21_IRAM = co21_IRAM[["fuente", "I_manual", "SigmaI"]] # CO(2-1) de class
co21_ALMA_beam10 = co21_ALMA_beam10[["fuente", "I_CO21_10_ALMA_K", "err_I_CO21_10_ALMA_K"]] # CO(2-1) con beam de CO(1-0)
hcn = hcn[["fuente", "I_manual", "SigmaI"]] # HCN de class
hco = hco[["fuente", "I_manual", "SigmaI"]] # HCO de class
hnc_ul = hnc_ul[["fuente", "I_ul_Jy", "I_ul_K"]] # HNC de class (solo límite superior) I < 3 * rms * \DeltaV
hnc = hnc[["fuente", "I_manual", "SigmaI"]] # HNC de class (solo se detecta en el centro)


# Ordenar por fuente para que coincidan las filas
co10_ALMA = co10_ALMA.sort_values("fuente").reset_index(drop=True)
co21_ALMA = co21_ALMA.sort_values("fuente").reset_index(drop=True)
co10_beam21 = co10_beam21.sort_values("fuente").reset_index(drop=True)
co10_beamdense = co10_beamdense.sort_values("fuente").reset_index(drop=True)
co10_IRAM = co10_IRAM.sort_values("fuente").reset_index(drop=True)
co21_IRAM = co21_IRAM.sort_values("fuente").reset_index(drop=True)
co21_ALMA_beam10 = co21_ALMA_beam10.sort_values("fuente").reset_index(drop=True)
hcn = hcn.sort_values("fuente").reset_index(drop=True)
hco = hco.sort_values("fuente").reset_index(drop=True)
hnc = hnc.sort_values("fuente").reset_index(drop=True)
hnc_ul = hnc_ul.sort_values("fuente").reset_index(drop=True)


## Ratio de gas termalizado $R_{21/10}$

Calculo $R_{21/10}=\frac{I_{CO(2-1)}}{I_{CO(1-0)}}$, dato que corresponde con el nivel de ionización del CO en la fuente. 

Calculo para cada fuente.

DEBO ASEGURARME DE UTILIZAR EL BEAM CORRECTO EN CADA CASO

In [2]:
from IPython.display import display
rows = []

# Primero con el CO(2-1) de IRAM y el CO(1-0) de ALMA con su beam. 

for i in range(len(co10_beam21)):
    fuente = co10_beam21["fuente"][i]

    # R21/10 usando I_manual
    R_2110_21IRAM     = co21_IRAM["I_manual"][i] / co10_beam21["I_CO10_21_ALMA_K"][i]
    err_R_2110_21IRAM = R_2110_21IRAM * np.sqrt((co21_IRAM["SigmaI"][i] / co21_IRAM["I_manual"][i])**2 +
                                                (co10_beam21["err_I_CO10_21_ALMA_K"][i] / co10_beam21["I_CO10_21_ALMA_K"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R2110_21IRAM"  : round(R_2110_21IRAM,     5),
        "err2110_21IRAM"  : round(err_R_2110_21IRAM, 5),
    })

df1= pd.DataFrame(rows)

# Luego con el CO(2-1) de ALMA con el beam de CO(1-0) y el CO(1-0) de IRAM. 

rows2 = []

for i in range (len(co10_IRAM)):
    fuente = co10_IRAM["fuente"][i]

    # R21/10 usando I_manual
    R_2110_21IRAM     = co21_ALMA_beam10["I_CO21_10_ALMA_K"][i] / co10_IRAM["I_manual"][i]
    err_R_2110_21IRAM = R_2110_21IRAM * np.sqrt((co21_ALMA_beam10["err_I_CO21_10_ALMA_K"][i] / co21_ALMA_beam10["I_CO21_10_ALMA_K"][i])**2 +
                                                (co10_IRAM["SigmaI"][i]  / co10_IRAM["I_manual"][i])**2)

    rows2.append({
        "fuente"   : fuente,
        "R2110_21ALMA"  : round(R_2110_21IRAM,     5),
        "err2110_21ALMA"  : round(err_R_2110_21IRAM, 5),
    })

df2 = pd.DataFrame(rows2)

out_df_CO = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_CO = out_df_CO.sort_values("fuente").reset_index(drop=True)
out_path = str(RESULTS_DIR / "W_RCO21_CO10.dat")
out_df_CO.to_csv(out_path, sep="\t", index=False)
display(out_df_CO)

fuente  R2110_21IRAM  err2110_21IRAM  R2110_21ALMA  err2110_21ALMA
NGC3110_CEN       0.66859         0.03578       0.64228         0.06505
NGC3110_NOR       1.34042         0.18785       0.32905         0.03782
NGC3110_SOU       0.87135         0.08601       0.65673         0.07442

## Ratio de HCN frente a CO (gas denso a difuso)

Ahora calculo el ratio entre flujo de HCN y CO, para comprobar si se trata de ULIRG o normal, además de la densidad de gas molecular denso

In [3]:
from IPython.display import display
rows = []
for i in range(len(co10_beamdense)):
    fuente = co10_beamdense["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HCN_CO     = hcn["I_manual"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i]
    err_R_HCN_CO = R_HCN_CO * np.sqrt((hcn["SigmaI"][i] / hcn["I_manual"][i])**2 +
                                       (co10_beamdense["err_I_CO10_DENSE_ALMA_K"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R_HCN_CO"  : round(R_HCN_CO,     5),
        "err_R_HCN_CO"  : round(err_R_HCN_CO, 5),
    })

order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HCN_CO = pd.DataFrame(rows)
out_df_HCN_CO["fuente"] = pd.Categorical(out_df_HCN_CO["fuente"], categories=order, ordered=True)
out_df_HCN_CO = out_df_HCN_CO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHCN_CO.dat")
out_df_HCN_CO.to_csv(out_path, sep="\t", index=False)
display(out_df_HCN_CO)

fuente  R_HCN_CO  err_R_HCN_CO
NGC3110_NOR   0.04034       0.01219
NGC3110_CEN   0.03354       0.00481
NGC3110_SOU   0.04654       0.00795

## Ratio de HCO frente a CO

Lo mismo con HCO+

In [4]:
from IPython.display import display
rows = []
for i in range(len(co10_beamdense)):
    fuente = co10_beamdense["fuente"][i]

    # R_HCO/CO10 usando I_manual
    R_HCO_CO     = hco["I_manual"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i]
    err_R_HCO_CO = R_HCO_CO * np.sqrt((hco["SigmaI"][i] / hco["I_manual"][i])**2 +
                                       (co10_beamdense["err_I_CO10_DENSE_ALMA_K"][i]  / co10_beamdense["I_CO10_DENSE_ALMA_K"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R_HCO_CO"  : round(R_HCO_CO,     5),
        "err_R_HCO_CO"  : round(err_R_HCO_CO, 5),
    })

order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HCO_CO = pd.DataFrame(rows)
out_df_HCO_CO["fuente"] = pd.Categorical(out_df_HCO_CO["fuente"], categories=order, ordered=True)
out_df_HCO_CO = out_df_HCO_CO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHCO_CO.dat")
out_df_HCO_CO.to_csv(out_path, sep="\t", index=False)
display(out_df_HCO_CO)

fuente  R_HCO_CO  err_R_HCO_CO
NGC3110_NOR   0.04152       0.01165
NGC3110_CEN   0.04144       0.00460
NGC3110_SOU   0.03673       0.00757

## Ratio de HNC frente a CO

Para HNC, al no detectarse de manera clara, se ha calculado un límite superior del flujo integrado. Se han utilizado 2 ecuaciones distintas: 

1. $I < 3 \ \text{rms} \ \Delta V$ 

2. $I_2 < 3 \ \text{rms} \ \sqrt{\Delta V \ \delta v}$ 

Se ha añadido el cálculo del ratio para el pointing central, ya que se detecta la línea. 

In [5]:
from IPython.display import display
rows = []
for i in range(len(co10_beamdense)):
    fuente = co10_beamdense["fuente"][i]

    # R_HNC/CO10 upper limit usando I_manual
    R_HNC_CO_ul     = hnc_ul["I_ul_K"][i] / co10_beamdense["I_CO10_DENSE_ALMA_K"][i]

    rows.append({
        "fuente"   : fuente,
        "R_HNC_CO_ul"  : round(R_HNC_CO_ul,     5),
    })

df1 = pd.DataFrame(rows)

rows2 = []
# R_HCN/CO10 usando I_manual solo para la detección del centro
R_HNC_CO     = hnc["I_manual"][0] / co10_beamdense["I_CO10_DENSE_ALMA_K"][0]
err_R_HNC_CO = R_HNC_CO * np.sqrt((hnc["SigmaI"][0] / hnc["I_manual"][0])**2 +
                                    (co10_beamdense["err_I_CO10_DENSE_ALMA_K"][0]  / co10_beamdense["I_CO10_DENSE_ALMA_K"][0])**2)

rows2.append({
    "fuente"   : co10_beamdense["fuente"][0],
    "R_HNC_CO"  : round(R_HNC_CO,     5),
    "err_R_HNC_CO"  : round(err_R_HNC_CO, 5),
})

df2 = pd.DataFrame(rows2)

out_df_HNC_CO = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HNC_CO["fuente"] = pd.Categorical(out_df_HNC_CO["fuente"], categories=order, ordered=True)
out_df_HNC_CO = out_df_HNC_CO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHNC_CO.dat")
out_df_HNC_CO.to_csv(out_path, sep="\t", index=False)
display(out_df_HNC_CO)

fuente  R_HNC_CO_ul  R_HNC_CO  err_R_HNC_CO
NGC3110_NOR      0.02226       NaN           NaN
NGC3110_CEN      0.00836   0.01946       0.00485
NGC3110_SOU      0.01517       NaN           NaN

## Ratio de HCN frente a HCO+

Estos datos son todos de las observaciones de IRAM centradas en la misma frecuencia, por lo que la escala de Jy/beam es la misma

In [6]:
from IPython.display import display
rows = []
for i in range(len(hco)):
    fuente = hco["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HCN_HCO     = hco["I_manual"][i] / hcn["I_manual"][i]
    err_R_HCN_HCO = R_HCN_HCO * np.sqrt((hcn["SigmaI"][i] / hcn["I_manual"][i])**2 +
                                          (hco["SigmaI"][i]  / hco["I_manual"][i])**2)

    rows.append({
        "fuente"   : fuente,
        "R_HCN_HCO"  : round(R_HCN_HCO,     5),
        "err_R_HCN_HCO"  : round(err_R_HCN_HCO, 5),
    })

order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HCN_HCO = pd.DataFrame(rows)
out_df_HCN_HCO["fuente"] = pd.Categorical(out_df_HCN_HCO["fuente"], categories=order, ordered=True)
out_df_HCN_HCO = out_df_HCN_HCO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHCN_HCO.dat")
out_df_HCN_HCO.to_csv(out_path, sep="\t", index=False)
display(out_df_HCN_HCO)

fuente  R_HCN_HCO  err_R_HCN_HCO
NGC3110_NOR    1.02925        0.41795
NGC3110_CEN    1.23553        0.20648
NGC3110_SOU    0.78921        0.20371

## Ratio de HNC frente a HCO+

Los datos de HNC son límites superiores en Norte y Sur, estimados a partir de $3 \cdot \sigma_{rms} \cdot \sqrt{\Delta V \delta V}$.

In [7]:
from IPython.display import display
rows = []
for i in range(len(hco)):
    fuente = hco["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HNC_HCO_ul     = hnc_ul["I_ul_K"][i] / hco["I_manual"][i]

    rows.append({
        "fuente"   : fuente,
        "R_HNC_HCO_ul"  : round(R_HNC_HCO_ul,     5),
    })

df1 = pd.DataFrame(rows)

rows2 = []
# R_HCN/HCO usando I_manual solo para la detección del centro
R_HNC_HCO     = hnc["I_manual"][0] / hco["I_manual"][0]
err_R_HNC_HCO = R_HNC_HCO * np.sqrt((hnc["SigmaI"][0] / hnc["I_manual"][0])**2 +
                                       (hco["SigmaI"][0]  / hco["I_manual"][0])**2)

rows2.append({
    "fuente"   : hco["fuente"][0],
    "R_HNC_HCO"  : round(R_HNC_HCO,     5),
    "err_R_HNC_HCO"  : round(err_R_HNC_HCO, 5),
})

df2 = pd.DataFrame(rows2)

out_df_HNC_HCO = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HNC_HCO["fuente"] = pd.Categorical(out_df_HNC_HCO["fuente"], categories=order, ordered=True)
out_df_HNC_HCO = out_df_HNC_HCO.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHNC_HCO.dat")
out_df_HNC_HCO.to_csv(out_path, sep="\t", index=False)
display(out_df_HNC_HCO)

fuente  R_HNC_HCO_ul  R_HNC_HCO  err_R_HNC_HCO
NGC3110_NOR       0.53616        NaN            NaN
NGC3110_CEN       0.20170    0.46972        0.12375
NGC3110_SOU       0.41301        NaN            NaN

## Ratio de HNC frente a HCN

Los datos de HNC son límites superiores en Norte y Sur

In [8]:
from IPython.display import display
rows = []
for i in range(len(hcn)):
    fuente = hcn["fuente"][i]

    # RHCN/CO10 usando I_manual
    R_HNC_HCN_ul     = hnc_ul["I_ul_K"][i] / hcn["I_manual"][i]

    rows.append({
        "fuente"   : fuente,
        "R_HNC_HCN_ul"  : round(R_HNC_HCN_ul,     5),
    })

df1 = pd.DataFrame(rows)

rows2 = []
# R_HNC/HCN usando I_manual solo para la detección del centro
R_HNC_HCN     = hnc["I_manual"][0] / hcn["I_manual"][0]
err_R_HNC_HCN = R_HNC_HCN * np.sqrt((hnc["SigmaI"][0] / hnc["I_manual"][0])**2 +
                                       (hcn["SigmaI"][0]  / hcn["I_manual"][0])**2)

rows2.append({
    "fuente"   : hcn["fuente"][0],
    "R_HNC_HCN"  : round(R_HNC_HCN,     5),
    "err_R_HNC_HCN"  : round(err_R_HNC_HCN, 5),
})

df2 = pd.DataFrame(rows2)

out_df_HNC_HCN = pd.merge(df1, df2, on="fuente", how="outer")
order = ["NGC3110_NOR", "NGC3110_CEN", "NGC3110_SOU"]
out_df_HNC_HCN["fuente"] = pd.Categorical(out_df_HNC_HCN["fuente"], categories=order, ordered=True)
out_df_HNC_HCN = out_df_HNC_HCN.sort_values("fuente").reset_index(drop=True)
out_path = os.path.join(results_dir, "W_RHNC_HCN.dat")
out_df_HNC_HCN.to_csv(out_path, sep="\t", index=False)
display(out_df_HNC_HCN)

fuente  R_HNC_HCN_ul  R_HNC_HCN  err_R_HNC_HCN
NGC3110_NOR       0.55184        NaN            NaN
NGC3110_CEN       0.24921    0.58035        0.16172
NGC3110_SOU       0.32595        NaN            NaN